# 09.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/05-hugging-face/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1

| model | parameters | float32 (×4 bytes) | float16 (×2 bytes) |
|---|---|---|---|
| DistilBERT | 66 million | 264 MB | 132 MB |
| BERT-base | 110 million | 440 MB | 220 MB |
| 3B LLM | 3 billion | 12 GB | 6 GB |
| 7B LLM | 7 billion | 28 GB | 14 GB |

The BERT models fit easily. A 3B LLM in float16 (6 GB) would leave almost nothing on an 8 GB laptop. That is why local LLMs are **quantised** to about 4 bits per parameter: 3 billion × 0.5 bytes ≈ 1.5–2 GB, which matches the 2 GB download of `llama3.2` in the setup guide.

In [2]:
for name, n in [("DistilBERT", 66e6), ("BERT-base", 110e6), ("3B LLM", 3e9), ("7B LLM", 7e9)]:
    print(f"{name:10}  float32 {n * 4 / 1e9:6.2f} GB   float16 {n * 2 / 1e9:6.2f} GB   4-bit {n * 0.5 / 1e9:5.2f} GB")

DistilBERT  float32   0.26 GB   float16   0.13 GB   4-bit  0.03 GB
BERT-base   float32   0.44 GB   float16   0.22 GB   4-bit  0.06 GB
3B LLM      float32  12.00 GB   float16   6.00 GB   4-bit  1.50 GB
7B LLM      float32  28.00 GB   float16  14.00 GB   4-bit  3.50 GB


## Exercise 2

In [3]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import torch, time
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
print("running on", {-1: "CPU", 0: "NVIDIA GPU", "mps": "Apple GPU"}[device])

emails = [
    "Hi, my order 18734 still hasn't arrived and it's been two weeks. Where is it?",
    "I'd like my money back for the lantern, it doesn't fit my shelf.",
    "Is the ceramic teapot dishwasher safe?",
    "The cake stand came smashed and nobody answered my last two emails. Very poor service.",
    "Can you tell me if the fairy lights work outdoors?",
    "Please refund the £24.50 for the cancelled doormat.",
    "Tracking says delivered but there is nothing at my door.",
    "Your courier left the parcel in the rain and everything inside is ruined. Unacceptable.",
]
truth = ["delivery", "refund", "product question", "complaint",
         "product question", "refund", "delivery", "complaint"]
topics = ["delivery", "refund", "product question", "complaint"]

def route(model_name):
    clf = pipeline("zero-shot-classification", model=model_name, device=device)
    start = time.time()
    out = clf(emails, candidate_labels=topics)
    pred = [o["labels"][0] for o in out]
    correct = sum(p == t for p, t in zip(pred, truth))
    return clf, out, pred, correct, time.time() - start

running on Apple GPU


In [4]:
emails += ["Thank you so much, the bunting arrived quickly and looks gorgeous!",
           "Just wanted to say your customer service was brilliant."]
truth += ["praise", "praise"]
topics = topics + ["praise"]
for template in ["This example is {}.", "This customer email is about {}."]:
    clf = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", device=device)
    pred = [o["labels"][0] for o in clf(emails, candidate_labels=topics, hypothesis_template=template)]
    print(f"{template!r:38} {sum(p == t for p, t in zip(pred, truth))} of {len(emails)} correct")
    print("   ", pred)

'This example is {}.'                  8 of 10 correct
    ['complaint', 'product question', 'product question', 'complaint', 'product question', 'refund', 'delivery', 'complaint', 'praise', 'praise']


'This customer email is about {}.'     3 of 10 correct
    ['product question', 'product question', 'complaint', 'complaint', 'praise', 'product question', 'product question', 'product question', 'praise', 'praise']


Ten emails, and one innocent-looking change of wording drops the score from 8 to 3 correct: the second template pulls almost everything into *product question*. The template is part of the model's input, so wording it differently changes the answers. This is a first taste of prompt engineering (09.8): small changes in how you ask can change results, and the only way to know is to test on labelled examples.

## Exercise 3

In [5]:
emails = [
    "Hi, my order 18734 still hasn't arrived and it's been two weeks. Where is it?",
    "I'd like my money back for the lantern, it doesn't fit my shelf.",
    "Is the ceramic teapot dishwasher safe?",
    "The cake stand came smashed and nobody answered my last two emails. Very poor service.",
    "Can you tell me if the fairy lights work outdoors?",
    "Please refund the £24.50 for the cancelled doormat.",
    "Tracking says delivered but there is nothing at my door.",
    "Your courier left the parcel in the rain and everything inside is ruined. Unacceptable.",
]
truth = ["delivery", "refund", "product question", "complaint",
         "product question", "refund", "delivery", "complaint"]
topics = ["delivery", "refund", "product question", "complaint"]

results = []
for name in ["typeform/distilbert-base-uncased-mnli", "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"]:
    clf, out, pred, correct, secs = route(name)
    params = sum(p.numel() for p in clf.model.parameters()) / 1e6
    results.append({"model": name.split("/")[1], "correct of 8": correct,
                    "seconds": round(secs, 1), "parameters (M)": round(params)})
pd.DataFrame(results)

,model,correct of 8,seconds,parameters (M)
0,distilbert-base-uncased-mnli,6,0.2,67
1,deberta-v3-base-zeroshot-v2.0,7,0.5,184


The larger DeBERTa model gets one more email right, at nearly three times the parameters and a slower run. With only 8 emails, one email is not a reliable difference: you would want a few hundred labelled examples before choosing. Bigger is not automatically better.

## Exercise 4

In [6]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
names = ["WHITE HANGING HEART T-LIGHT HOLDER", "JUMBO BAG RED RETROSPOT", "REGENCY CAKESTAND 3 TIER",
         "PACK OF 72 RETROSPOT CAKE CASES", "ASSORTED COLOUR BIRD ORNAMENT", "LUNCH BAG SUKI DESIGN",
         "SET OF 3 CAKE TINS PANTRY DESIGN", "NATURAL SLATE HEART CHALKBOARD", "HAND WARMER BABUSHKA",
         "PAPER CHAIN KIT 50'S CHRISTMAS"]
rows = [(n, len(n.split()), len(tok.tokenize(n)), tok.tokenize(n)) for n in names]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows, columns=["name", "words", "tokens", "pieces"]).sort_values("tokens", ascending=False)

,name,words,tokens,pieces
3,PACK OF 72 RETROSPOT CAKE CASES,6,8,"[pack, of, 72, retro, ##sp, ##ot, cake, cases]"
6,SET OF 3 CAKE TINS PANTRY DESIGN,7,8,"[set, of, 3, cake, tin, ##s, pantry, design]"
0,WHITE HANGING HEART T-LIGHT HOLDER,5,7,"[white, hanging, heart, t, -, light, holder]"
1,JUMBO BAG RED RETROSPOT,4,7,"[ju, ##mbo, bag, red, retro, ##sp, ##ot]"
4,ASSORTED COLOUR BIRD ORNAMENT,4,7,"[ass, ##orted, colour, bird, or, ##name, ##nt]"
9,PAPER CHAIN KIT 50'S CHRISTMAS,5,7,"[paper, chain, kit, 50, ', s, christmas]"
2,REGENCY CAKESTAND 3 TIER,4,6,"[regency, cakes, ##tan, ##d, 3, tier]"
5,LUNCH BAG SUKI DESIGN,4,5,"[lunch, bag, su, ##ki, design]"
7,NATURAL SLATE HEART CHALKBOARD,4,5,"[natural, slate, heart, chalk, ##board]"
8,HAND WARMER BABUSHKA,3,5,"[hand, warmer, babu, ##sh, ##ka]"


Shop-specific words are split into pieces: *retrospot* → *retro, ##sp, ##ot*; *cakestand* → *cakes, ##tan, ##d*; *babushka* → *babu, ##sh, ##ka*. So are some ordinary but less common words (*jumbo*, *assorted*, *ornament*), while everyday words (*bag*, *heart*, *holder*) stay whole. Two consequences: the model has weaker knowledge of these words (it has to assemble their meaning from fragments), and with paid LLM APIs you are charged **per token**, so unusual vocabulary costs more.